# 2a — Preprocessing: yellow taxi trip records

**Input:** `data/landing/tlc/yellow_tripdata_YYYY-MM.parquet` (18 monthly files, Jan 2023 – Jun 2024)

**Outputs:**

| File | Contents |
|---|---|
| `data/raw/trips_clean.parquet` | Cleaned citywide trip records, partitioned by month. Read by notebook 3. |
| `data/curated/taxi_airport_hourly.parquet` | Table A — one row per `(pickup_date, pickup_hour, airport)` for JFK and LGA. |
| `data/curated/preprocessing_counts_taxi.csv` | Record count remaining after each filter. |
| `data/curated/shapes_taxi.json` | Headline shapes and retention figures. |

## Flow

1. Load and harmonise the eighteen monthly files.
2. Define the business-rule filters as an ordered list of `(label, predicate)` pairs.
3. Count the records remaining after each filter, in one pass.
4. Apply the filters and write the cleaned citywide dataset. Profile the records
   whose metadata was not recorded.
5. Subset to JFK and LaGuardia pickups, after cross-checking the zone identifiers
   against the meter-entered rate code.
6. Aggregate to Table A on a complete hourly spine, flagging the daylight-saving hours.
7. Validate and write.

Cleaning is applied to the full citywide dataset *before* the airport subset is taken,
so that notebook 3 can describe the whole distribution. Records are removed only where a
documented business rule or a logical impossibility is violated; records that are merely
extreme, or whose metadata is unrecorded, are retained and flagged.


In [1]:
"""Preprocessing of the TLC yellow taxi trip records.

Reads the raw monthly parquet files, harmonises their schemas, applies
business-rule filters to the full citywide dataset, and aggregates the airport
subset to hourly counts on the grid used by every downstream notebook.
"""

import json
import sys
from functools import reduce
from operator import and_
from pathlib import Path

from pyspark.sql import functions as F

# Session configuration, schema normalisation, the verified zone identifiers,
# and the reporting helpers shared with notebook 2b live in
# `scripts/spark_utils.py` so that every notebook loads the data identically
# and the logic can be linted as ordinary source.
sys.path.append(str(Path("..") / "scripts"))
from spark_utils import (  # noqa: E402
    ARRIVAL_AIRPORTS,
    JFK_ZONE,
    LGA_ZONE,
    count_waterfall,
    create_spark_session,
    hour_spine,
    load_trips,
    print_waterfall,
)

# --- Paths -----------------------------------------------------------------
# Notebook is expected to run from `notebooks/`.
PROJECT_ROOT = Path("..").resolve()
LANDING_DIR = PROJECT_ROOT / "data" / "landing" / "tlc"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CURATED_DIR = PROJECT_ROOT / "data" / "curated"

for directory in (RAW_DIR, CURATED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# --- Study window ----------------------------------------------------------
# Inclusive of the start date, exclusive of the end date.
WINDOW_START = "2023-01-01"
WINDOW_END = "2024-07-01"
WINDOW_DAYS = 547
EXPECTED_ROWS = WINDOW_DAYS * 24 * len(ARRIVAL_AIRPORTS)  # 26,256

# --- Taxi zones ------------------------------------------------------------
AIRPORT_ZONES = {JFK_ZONE: "JFK", LGA_ZONE: "LGA"}
# EWR (zone 1) is deliberately excluded. Yellow taxis may drop off at Newark but
# may not pick up there, so EWR enters this study only through the flight-side
# features built in notebook 2b.

# The airport codes must agree with the ones notebook 2b keys its flight table
# on, or the join in 2c silently drops rows instead of failing.
assert tuple(AIRPORT_ZONES.values()) == ARRIVAL_AIRPORTS

In [2]:
spark = create_spark_session(app_name="MAST30034 — taxi preprocessing")
spark.sparkContext.setLogLevel("WARN")
spark.version

your 131072x1 screen size is bogus. expect trouble
26/08/16 19:00:53 WARN Utils: Your hostname, iphone resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/16 19:00:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/16 19:00:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.1'

## Step 1 — Load and harmonise the monthly files

`load_trips` reads every `yellow_tripdata_*.parquet` file in the landing directory,
passes each through `normalise_schema`, and unions them with `unionByName` so that a
difference in column ordering cannot misalign the frames.

`normalise_schema` reconciles the two schema differences across the eighteen months: the
`airport_fee` / `Airport_fee` casing difference, and the integer-width drift in the
identifier and count columns.


In [3]:
trips_raw = load_trips(spark, LANDING_DIR)
print(f"{len(trips_raw.columns)} columns after normalisation")

19 columns after normalisation


In [4]:
# Trip duration and implied speed, derived before any filtering so that the
# count waterfall in Step 3 can use them.
trips_raw = (
    trips_raw
    .withColumn(
        "trip_duration_s",
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime"),
    )
    .withColumn(
        "implied_mph",
        F.when(
            F.col("trip_duration_s") > 0,
            F.col("trip_distance") / (F.col("trip_duration_s") / 3600.0),
        ),
    )
)

## Step 2 — Filter definitions

Nine filters are defined as an ordered list of `(label, predicate)` pairs. The labels are
reused by the count waterfall in Step 3 and by the preprocessing table in the report, so
the two stay in step. Thresholds taken from the TLC fare schedule and data dictionary are
held in named constants above the list.

Predicates admit nulls explicitly wherever a null is meant to survive: `.where()` drops
rows whose predicate evaluates to null, so a bare `.isin()` would silently delete the
records with an unrecorded rate code or payment type rather than retaining them.


In [5]:
# Initial unit charge on the standard city rate (Rate Code 1), in force from
# 19 December 2022 [3].
MIN_METERED_FARE = 3.00

MAX_IMPLIED_MPH = 80.0
MIN_DURATION_S = 60
MAX_DURATION_S = 6 * 3600

# Codes defined by the data dictionary [2]. Rate code 99 is the dictionary's
# own "Null/unknown" sentinel and payment type 0 is a Flex Fare trip; only
# values the dictionary does not define at all are removed.
RATECODE_STANDARD = 1
RATECODE_JFK_FLAT = 2
RATECODE_UNKNOWN = 99
DOCUMENTED_RATECODES = [1, 2, 3, 4, 5, 6, RATECODE_UNKNOWN]
DOCUMENTED_PAYMENT_TYPES = [0, 1, 2, 3, 4, 5, 6]
PAYMENT_CARD = 1
PAYMENT_FLEX_FARE = 0

# Ordered list of (label, predicate), consumed by the waterfall in Step 3 and
# applied as a conjunction in Step 4.
FILTERS = [
    (
        "Pickup within study window",
        (F.col("tpep_pickup_datetime") >= F.lit(WINDOW_START).cast("timestamp"))
        & (F.col("tpep_pickup_datetime") < F.lit(WINDOW_END).cast("timestamp")),
    ),
    (
        "Drop-off after pickup",
        F.col("trip_duration_s") > 0,
    ),
    (
        "Duration between 1 min and 6 h",
        (F.col("trip_duration_s") >= MIN_DURATION_S)
        & (F.col("trip_duration_s") <= MAX_DURATION_S),
    ),
    (
        "Positive trip distance",
        F.col("trip_distance") > 0,
    ),
    (
        "Positive fare and total",
        (F.col("fare_amount") > 0) & (F.col("total_amount") > 0),
    ),
    (
        "Metered fare at or above initial charge",
        # `eqNullSafe` so that a missing rate code yields False rather than
        # null, which `.where()` would drop.
        ~F.col("ratecodeid").eqNullSafe(RATECODE_STANDARD)
        | (F.col("fare_amount") >= MIN_METERED_FARE),
    ),
    (
        "Implied speed below 80 mph",
        F.col("implied_mph") < MAX_IMPLIED_MPH,
    ),
    (
        "Documented rate code",
        F.col("ratecodeid").isNull()
        | F.col("ratecodeid").isin(DOCUMENTED_RATECODES),
    ),
    (
        "Documented payment type",
        F.col("payment_type").isNull()
        | F.col("payment_type").isin(DOCUMENTED_PAYMENT_TYPES),
    ),
]

## Step 3 — Record counts after each filter

`count_waterfall` evaluates the cumulative conjunction of the predicates in a single pass
— summing `predicate_1`, then `predicate_1 AND predicate_2`, and so on — which gives the
sequential post-filter counts without nine full scans of 58.6M rows. It lives in
`scripts/spark_utils.py` because notebook 2b builds the same table from the flight data.

It counts a null predicate as an exclusion, matching the behaviour of `.where()`.


In [6]:
waterfall = count_waterfall(trips_raw, FILTERS)
print_waterfall(waterfall)

Raw ingest                                   58,642,319  removed          —  (100.00% of raw)
Pickup within study window                   58,642,192  removed        127  (100.00% of raw)
Drop-off after pickup                        58,620,446  removed     21,746  ( 99.96% of raw)
Duration between 1 min and 6 h               57,922,347  removed    698,099  ( 98.77% of raw)
Positive trip distance                       57,243,639  removed    678,708  ( 97.61% of raw)
Positive fare and total                      56,642,200  removed    601,439  ( 96.59% of raw)
Metered fare at or above initial charge      56,641,660  removed        540  ( 96.59% of raw)
Implied speed below 80 mph                   56,637,949  removed      3,711  ( 96.58% of raw)
Documented rate code                         56,637,949  removed          0  ( 96.58% of raw)
Documented payment type                      56,637,949  removed          0  ( 96.58% of raw)


In [7]:
# Written to CSV for the report's preprocessing table.
raw_total = waterfall[0][1]
rows = [
    {
        "step": label,
        "rows": remaining,
        "removed": removed,
        "pct_of_raw": round(100 * remaining / raw_total, 3),
    }
    for label, remaining, removed in waterfall
]

counts_path = CURATED_DIR / "preprocessing_counts_taxi.csv"
spark.createDataFrame(rows).coalesce(1).toPandas().to_csv(counts_path, index=False)
print(f"Written to {counts_path}")

Written to /home/tavish/projects/project-1-individual-SavvyHack/data/curated/preprocessing_counts_taxi.csv


## Step 4 — Apply the filters and persist the cleaned citywide dataset

The filter conjunction is applied, four flag columns are added for the retained-but-
unrecorded records, and the result is written partitioned by month. This is the dataset
notebook 3 reads for the distribution and outlier analysis.


In [8]:
trips_clean = (
    trips_raw
    .where(reduce(and_, (predicate for _, predicate in FILTERS)))
    .withColumn("pickup_month", F.date_format("tpep_pickup_datetime", "yyyy-MM"))
    # Flags for unrecorded metadata. A null rate code and a 99 both mean
    # "not recorded", so they are flagged identically.
    .withColumn(
        "ratecode_missing",
        F.col("ratecodeid").isNull()
        | (F.col("ratecodeid") == RATECODE_UNKNOWN),
    )
    .withColumn("passenger_count_missing", F.col("passenger_count").isNull())
    .withColumn("payment_type_missing", F.col("payment_type").isNull())
    .withColumn(
        "flex_fare",
        F.coalesce(F.col("payment_type") == PAYMENT_FLEX_FARE, F.lit(False)),
    )
)

(
    trips_clean
    .write
    .mode("overwrite")
    .partitionBy("pickup_month")
    .parquet(str(RAW_DIR / "trips_clean.parquet"))
)
print("Cleaned citywide dataset written.")

26/08/16 19:01:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/08/16 19:01:29 WARN DAGScheduler: Broadcasting large task binary with size 1015.8 KiB


Cleaned citywide dataset written.


In [9]:
# Read back from disk so that downstream stages do not re-run the filter
# chain over all eighteen monthly files.
trips_clean = spark.read.parquet(str(RAW_DIR / "trips_clean.parquet"))
clean_total = trips_clean.count()
assert clean_total == waterfall[-1][1], (
    f"Written {clean_total:,} rows but waterfall predicted {waterfall[-1][1]:,}"
)
print(f"{clean_total:,} rows verified on disk")

56,637,949 rows verified on disk


### Step 4a — Profile of the records with unrecorded metadata

Breaks the retained records with a missing rate code down by `vendorid` and by pickup-zone
group (airport zone, unknown zone, or other NYC zone), and reports the citywide rate for
both the rate code and the passenger count.


In [10]:
UNKNOWN_ZONES = [264, 265]  # "Unknown" and "Outside of NYC / N.A." [4]

missing_profile = (
    trips_clean
    .where(F.col("ratecode_missing"))
    .groupBy(
        "vendorid",
        F.when(F.col("pulocationid").isin(list(AIRPORT_ZONES)), "airport zone")
        .when(F.col("pulocationid").isin(UNKNOWN_ZONES), "unknown zone")
        .otherwise("other NYC zone")
        .alias("zone_group"),
    )
    .agg(
        F.count("*").alias("trips"),
        F.avg(F.col("passenger_count").isNull().cast("double"))
        .alias("share_passenger_null"),
        F.avg(F.col("flex_fare").cast("double")).alias("share_flex_fare"),
    )
    .orderBy(F.desc("trips"))
)
missing_profile.show(truncate=False)

missing_total = trips_clean.where(F.col("ratecode_missing")).count()
passenger_missing_total = trips_clean.where(F.col("passenger_count_missing")).count()
print(f"Unrecorded rate code:      {missing_total:,} "
      f"({100 * missing_total / clean_total:.2f}% of cleaned trips)")
print(f"Unrecorded passenger count: {passenger_missing_total:,} "
      f"({100 * passenger_missing_total / clean_total:.2f}% of cleaned trips)")

+--------+--------------+-------+--------------------+------------------+
|vendorid|zone_group    |trips  |share_passenger_null|share_flex_fare   |
+--------+--------------+-------+--------------------+------------------+
|2       |other NYC zone|2201776|1.0                 |1.0               |
|1       |other NYC zone|959434 |0.590909848931766   |0.590909848931766 |
|1       |airport zone  |15574  |0.9976884551175035  |0.9976884551175035|
|6       |unknown zone  |6104   |1.0                 |1.0               |
|1       |unknown zone  |2618   |0.806340718105424   |0.806340718105424 |
|2       |unknown zone  |1051   |1.0                 |1.0               |
|2       |airport zone  |370    |1.0                 |1.0               |
+--------+--------------+-------+--------------------+------------------+



Unrecorded rate code:      3,186,927 (5.63% of cleaned trips)
Unrecorded passenger count: 2,793,889 (4.93% of cleaned trips)


## Step 5 — Airport subset

Pickups at JFK (zone 132) and LaGuardia (zone 138) are selected on the **pickup** zone
alone, since a trip *to* an airport is not part of the target.

Before the subset is taken, the zone identifiers are cross-checked against Rate Code 2 —
the JFK flat fare, which is entered at the meter independently of the GPS-derived
`PULocationID`. The share of Rate Code 2 trips touching zone 132 at either end is reported.


In [11]:
# Cross-check of the zone identifier against the meter-entered rate code.
ratecode_2 = trips_clean.where(F.col("ratecodeid") == RATECODE_JFK_FLAT)
validation = ratecode_2.agg(
    F.count("*").alias("ratecode_2_trips"),
    F.avg(
        F.when(
            (F.col("pulocationid") == JFK_ZONE)
            | (F.col("dolocationid") == JFK_ZONE),
            1.0,
        ).otherwise(0.0)
    ).alias("share_touching_zone_132"),
).collect()[0]

print(f"Rate Code 2 trips:            {validation['ratecode_2_trips']:,}")
print(f"Share touching zone 132:      {validation['share_touching_zone_132']:.4f}")

Rate Code 2 trips:            2,019,422
Share touching zone 132:      0.9619


In [12]:
airport_trips = (
    trips_clean
    .where(F.col("pulocationid").isin(list(AIRPORT_ZONES)))
    .withColumn(
        "airport",
        F.when(F.col("pulocationid") == JFK_ZONE, F.lit("JFK"))
        .otherwise(F.lit("LGA")),
    )
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    # Read four more times below, so cached once here.
    .cache()
)

airport_total = airport_trips.count()
print(f"Airport pickups: {airport_total:,} "
      f"({100 * airport_total / clean_total:.2f}% of cleaned trips)")
airport_trips.groupBy("airport").count().show()

Airport pickups: 4,653,181 (8.22% of cleaned trips)
+-------+-------+
|airport|  count|
+-------+-------+
|    JFK|2753043|
|    LGA|1900138|
+-------+-------+



## Step 6 — Aggregate to Table A

One row per `(pickup_date, pickup_hour, airport)`, carrying the pickup count, revenue sums
and means, trip shape, and payment composition.

Three statistics are computed over a subset of the hour's trips and each is paired with
its own denominator column, so that a null is never mistaken for a zero downstream:

| Statistic | Computed over | Denominator column |
|---|---|---|
| `share_flat_fare` | trips whose rate code was recorded | `n_ratecode_known` |
| `mean_passengers` | trips whose passenger count was recorded | `n_passenger_known` |
| `mean_tip_ratio` | card payments only, since cash tips are not recorded [2] | `n_card_trips` |

Revenue is carried as sums as well as means, so that an earnings-shaped target is
available to notebook 4 without a second pass over the trip records.


In [13]:
hourly = airport_trips.groupBy("pickup_date", "pickup_hour", "airport").agg(
    F.count("*").alias("n_pickups"),

    # --- Fare and revenue --------------------------------------------------
    F.sum("fare_amount").alias("sum_fare_amount"),
    F.sum("total_amount").alias("sum_total_amount"),
    F.avg("fare_amount").alias("mean_fare"),
    F.expr("percentile_approx(fare_amount, 0.5)").alias("median_fare"),
    F.avg("total_amount").alias("mean_total"),

    # --- Trip shape --------------------------------------------------------
    F.avg("trip_distance").alias("mean_distance_mi"),
    F.avg(F.col("trip_duration_s") / 60.0).alias("mean_duration_min"),

    # --- Passenger count, over the trips where it was recorded -------------
    F.sum(F.when(~F.col("passenger_count_missing"), 1).otherwise(0))
    .alias("n_passenger_known"),
    F.avg("passenger_count").alias("mean_passengers"),

    # --- Rate code, over the trips where it was recorded -------------------
    # The third branch of the share is left unset, so a missing rate code
    # contributes to neither the numerator nor the denominator.
    F.sum(F.when(~F.col("ratecode_missing"), 1).otherwise(0))
    .alias("n_ratecode_known"),
    F.avg(
        F.when(F.col("ratecodeid") == RATECODE_JFK_FLAT, 1.0)
        .when(~F.col("ratecode_missing"), 0.0)
    ).alias("share_flat_fare"),

    # --- Payment -----------------------------------------------------------
    F.avg(F.col("flex_fare").cast("double")).alias("share_flex_fare"),
    F.sum(F.when(F.col("payment_type") == PAYMENT_CARD, 1).otherwise(0))
    .alias("n_card_trips"),
    # Tips are recorded for card payments only [2], so both the total and the
    # mean are taken over card trips alone.
    F.sum(
        F.when(F.col("payment_type") == PAYMENT_CARD, F.col("tip_amount"))
    ).alias("sum_tip_amount"),
    F.avg(
        F.when(
            F.col("payment_type") == PAYMENT_CARD,
            F.col("tip_amount") / F.col("fare_amount"),
        )
    ).alias("mean_tip_ratio"),
)

non_empty_hours = hourly.count()
print(f"{non_empty_hours:,} non-empty airport-hours")

25,076 non-empty airport-hours


### Hours with no pickups

An airport-hour with zero pickups produces no group and is absent from the aggregation
above. A complete spine of every `(date, hour, airport)` combination is built by
`hour_spine` — the same helper notebook 2b uses, so the two tables share a grid — and the
aggregation is left-joined onto it.

Counts and sums are filled with zero; means and shares are left null, since the mean fare
of no trips is undefined rather than zero.


In [14]:
spine = hour_spine(
    spark,
    WINDOW_START,
    WINDOW_END,
    airports=ARRIVAL_AIRPORTS,
    date_col="pickup_date",
    hour_col="pickup_hour",
)
assert spine.count() == EXPECTED_ROWS, "Spine does not cover the study window"

# Counts and revenue sums are zero-filled; means and shares stay null.
ZERO_FILL = [
    "n_pickups",
    "n_card_trips",
    "n_ratecode_known",
    "n_passenger_known",
    "sum_fare_amount",
    "sum_total_amount",
    "sum_tip_amount",
]

table_a = (
    spine
    .join(hourly, ["pickup_date", "pickup_hour", "airport"], how="left")
    .fillna(0, subset=ZERO_FILL)
)

### Daylight saving

Timestamps are New York wall-clock time, so the spine contains two kinds of anomalous
hour: the 02:00 that does not exist on the spring-forward dates, which the join fills with
zero pickups, and the 01:00 that occurs twice on the fall-back date, which therefore holds
two clock hours of trips.

Six rows of 26,256 are affected. They are flagged rather than deleted, so that the
modelling notebook can exclude them explicitly.


In [15]:
DST_SPRING_FORWARD = ["2023-03-12", "2024-03-10"]  # 02:00 does not exist
DST_FALL_BACK = ["2023-11-05"]                      # 01:00 occurs twice

table_a = table_a.withColumn(
    "dst_anomaly",
    (
        F.col("pickup_date").cast("string").isin(DST_SPRING_FORWARD)
        & (F.col("pickup_hour") == 2)
    )
    | (
        F.col("pickup_date").cast("string").isin(DST_FALL_BACK)
        & (F.col("pickup_hour") == 1)
    ),
).cache()

table_a.where(F.col("dst_anomaly")).select(
    "pickup_date", "pickup_hour", "airport", "n_pickups"
).orderBy("pickup_date", "airport").show()

+-----------+-----------+-------+---------+
|pickup_date|pickup_hour|airport|n_pickups|
+-----------+-----------+-------+---------+
| 2023-03-12|          2|    JFK|        0|
| 2023-03-12|          2|    LGA|        0|
| 2023-11-05|          1|    JFK|       47|
| 2023-11-05|          1|    LGA|        0|
| 2024-03-10|          2|    JFK|        0|
| 2024-03-10|          2|    LGA|        0|
+-----------+-----------+-------+---------+



## Step 7 — Validate and write

Six assertions on Table A — grid completeness, key uniqueness, trip conservation through
the grouping and join, the two denominator invariants, and revenue reconciliation —
followed by the parquet write and the shapes JSON.


In [16]:
# 1. Every airport-hour in the window is present exactly once.
assert table_a.count() == EXPECTED_ROWS
assert table_a.dropDuplicates(
    ["pickup_date", "pickup_hour", "airport"]
).count() == EXPECTED_ROWS

# 2. No trip was lost or duplicated by the grouping and the join.
assert table_a.agg(F.sum("n_pickups")).collect()[0][0] == airport_total

# 3. The flat-fare share is defined in exactly the hours where a rate code
#    was observed.
mismatched = table_a.where(
    F.col("share_flat_fare").isNull() != (F.col("n_ratecode_known") == 0)
).count()
assert mismatched == 0, f"{mismatched} rows disagree on the rate code denominator"

# 4. The same, for the passenger count.
mismatched = table_a.where(
    F.col("mean_passengers").isNull() != (F.col("n_passenger_known") == 0)
).count()
assert mismatched == 0, f"{mismatched} rows disagree on the passenger denominator"

# 5. Revenue survived the aggregation and the zero-fill. Compared with a
#    tolerance, since a distributed sum of doubles is not associative.
revenue_table = table_a.agg(F.sum("sum_total_amount")).collect()[0][0]
revenue_trips = airport_trips.agg(F.sum("total_amount")).collect()[0][0]
assert abs(revenue_table - revenue_trips) < 1.0, (
    f"Revenue disagrees: {revenue_table:,.2f} vs {revenue_trips:,.2f}"
)
print(f"Airport revenue reconciled: ${revenue_table:,.2f}")

# 6. Genuine zero-demand hours, excluding the DST artefacts.
zero_hours = table_a.where(
    (F.col("n_pickups") == 0) & (~F.col("dst_anomaly"))
).count()
print(f"Genuine zero-pickup airport-hours: {zero_hours} of {EXPECTED_ROWS}")
print("All checks passed.")

Airport revenue reconciled: $354,004,177.23
Genuine zero-pickup airport-hours: 1175 of 26256
All checks passed.


In [17]:
(
    table_a
    .orderBy("pickup_date", "pickup_hour", "airport")
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(CURATED_DIR / "taxi_airport_hourly.parquet"))
)

table_a.orderBy("pickup_date", "pickup_hour", "airport").show(6, truncate=False)

+-----------+-----------+-------+---------+------------------+-----------------+------------------+-----------+-----------------+------------------+------------------+-----------------+------------------+----------------+-------------------+--------------------+------------+------------------+-------------------+-----------+
|pickup_date|pickup_hour|airport|n_pickups|sum_fare_amount   |sum_total_amount |mean_fare         |median_fare|mean_total       |mean_distance_mi  |mean_duration_min |n_passenger_known|mean_passengers   |n_ratecode_known|share_flat_fare    |share_flex_fare     |n_card_trips|sum_tip_amount    |mean_tip_ratio     |dst_anomaly|
+-----------+-----------+-------+---------+------------------+-----------------+------------------+-----------+-----------------+------------------+------------------+-----------------+------------------+----------------+-------------------+--------------------+------------+------------------+-------------------+-----------+
|2023-01-01 |0     

In [18]:
# Airport-side shapes, recorded alongside the citywide waterfall. Retained-
# but-flagged records are counted at both scales.
airport_ratecode_known = table_a.agg(F.sum("n_ratecode_known")).collect()[0][0]
airport_passenger_known = table_a.agg(F.sum("n_passenger_known")).collect()[0][0]

shapes = {
    "raw_ingest_trips": raw_total,
    "cleaned_citywide_trips": clean_total,
    "removed_trips": raw_total - clean_total,
    "removed_pct": round(100 * (raw_total - clean_total) / raw_total, 3),
    "airport_pickups": airport_total,
    "airport_pickup_pct_of_clean": round(100 * airport_total / clean_total, 3),
    "non_empty_airport_hours": non_empty_hours,
    "table_a_rows": EXPECTED_ROWS,
    "genuine_zero_pickup_hours": zero_hours,
    "citywide_missing_ratecode": missing_total,
    "citywide_missing_passenger_count": passenger_missing_total,
    "airport_missing_ratecode": airport_total - airport_ratecode_known,
    "airport_missing_passenger_count": airport_total - airport_passenger_known,
    "ratecode_2_trips": validation["ratecode_2_trips"],
    "ratecode_2_share_touching_jfk": round(
        float(validation["share_touching_zone_132"]), 4
    ),
}
with open(CURATED_DIR / "shapes_taxi.json", "w") as handle:
    json.dump(shapes, handle, indent=2)

shapes

{'raw_ingest_trips': 58642319,
 'cleaned_citywide_trips': 56637949,
 'removed_trips': 2004370,
 'removed_pct': 3.418,
 'airport_pickups': 4653181,
 'airport_pickup_pct_of_clean': 8.216,
 'non_empty_airport_hours': 25076,
 'table_a_rows': 26256,
 'genuine_zero_pickup_hours': 1175,
 'citywide_missing_ratecode': 3186927,
 'citywide_missing_passenger_count': 2793889,
 'airport_missing_ratecode': 15944,
 'airport_missing_passenger_count': 15908,
 'ratecode_2_trips': 2019422,
 'ratecode_2_share_touching_jfk': 0.9619}

In [19]:
spark.stop()

## References

1. NYC Taxi and Limousine Commission. *TLC Trip Record Data*.
   <https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page>
2. NYC Taxi and Limousine Commission. *Yellow Trips Data Dictionary*.
   <https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf>
3. NYC Taxi and Limousine Commission. *Industry Notice #22-02: Taxicab Rate of Fare
   Increase*, effective 19 December 2022.
4. NYC Taxi and Limousine Commission. *Taxi Zone Lookup Table*.
   <https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv>
